# Clean Raw Mercury

Reads from this notebook's `input/` folder and writes to its `output/` and `reports/` folders (all siblings of this notebook file). Each of the four sections below is self-contained (its own schema constants, `clean()` function, and `df_raw_*`/`df_cleaned_*` variables) and can be re-run independently without clobbering another section's results.

## Imports & shared helpers

`find_default_input`, `month_tag_from_filename`, `write_report`, and `collapse_duplicate_rows` are identical in shape across all four source notebooks. Here they're defined once, parameterized by `directory`/`prefix`/`filename_re` (and `reports_dir`/dataframes for the report writer, `ls_cols`/`sum_cols`/`sort_cols` for the dedup collapser) instead of closing over notebook-global constants, and each of the four sections below calls these same functions with its own values.

Schema constants, `clean()` logic, and sort/rename rules genuinely differ per dataset and are kept written out separately in each section rather than hidden behind a generic abstraction.

In [1]:
import csv
import io
import re
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
REPORTS_DIR = NOTEBOOK_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

### `find_default_input`

Looks for a single `{prefix}_yyyy-mm-dd.txt` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.

In [2]:
def find_default_input(directory: Path, prefix: str, filename_re: re.Pattern) -> Path:
    matches = sorted(p for p in directory.glob(f"{prefix}_*.txt") if filename_re.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No {prefix}_yyyy-mm-dd.txt file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]

### `month_tag_from_filename`

The output name has the format `{prefix}_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one). Independent of any raw `yearMonth` data column.

In [3]:
def month_tag_from_filename(path: Path, prefix: str, filename_re: re.Pattern) -> str:
    match = filename_re.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern {prefix}_yyyy-mm-dd.txt")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"

### `write_report`

Writes the plain-text summary report: input filename, raw row/duplicate counts, `month_tag`, blank/missing-key rows dropped, duplicate rows collapsed, cleaned row/duplicate counts, output filename, then (after three blank lines) `df_cleaned.describe()`. Duplicate counts use pandas' default `duplicated()` (`keep="first"`) — the number of rows that would go away if the dataframe were deduplicated. `rows_before_collapse` is the row count right after `clean_*()` but before `collapse_duplicate_rows()`, which is what lets the report split "blank/missing-key rows dropped" from "duplicate rows collapsed" instead of lumping both into one raw-vs-cleaned delta. Saved to `reports_dir` as `{prefix}_{month_tag}_report.txt`. Returns `(report_path, report_text)` so the calling cell can print/inspect it.

In [ ]:
def write_report(
    reports_dir: Path,
    prefix: str,
    month_tag: str,
    input_path: Path,
    df_raw: pd.DataFrame,
    df_cleaned: pd.DataFrame,
    output_path: Path,
    rows_before_collapse: int,
) -> tuple[Path, str]:
    report_lines = [
        f"Input file: {input_path.name}",
        f"Raw row count: {len(df_raw)}",
        f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
        "=======================================================================",
        f"Month tag: {month_tag}",
        f"Blank/missing-key rows dropped: {len(df_raw) - rows_before_collapse}",
        f"Duplicate rows collapsed: {rows_before_collapse - len(df_cleaned)}",
        "=======================================================================",
        f"Cleaned row count: {len(df_cleaned)}",
        f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
        f"Output file: {output_path.name}",
    ]
    report_text = "\n".join(report_lines) + "\n"
    report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

    reports_dir.mkdir(parents=True, exist_ok=True)
    report_path = reports_dir / f"{prefix}_{month_tag}_report.txt"
    report_path.write_text(report_text, encoding="utf-8")

    return report_path, report_text

### `read_semicolon_csv_protecting_backslashes`

Used by the DailyEvents/MonthlyEvents sections, both of which need `engine="python", escapechar="\\"` to parse genuine `\"..\"` escapes around quoted phrases inside `SearchTerm`/`ContentTitle` (e.g. `\"daily wellness check-in\"`). Left unguarded, `escapechar` strips *every* backslash it precedes, not just ones before a quote — so a literal backslash in `CompanyCode`/`CompanyName` (e.g. a client named `TBWA\RAAD`) would be silently corrupted to `TBWARAAD`. This pre-processes the raw text to double any backslash *not* immediately followed by `"`, so `escapechar` only ever consumes genuine `\"` sequences and every other backslash survives intact.

In [5]:
def read_semicolon_csv_protecting_backslashes(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Protect literal backslashes that aren't a genuine CSV \" escape by doubling them,
    # so escapechar only ever consumes actual \" sequences below.
    protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

    return pd.read_csv(
        io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
        dtype=str, keep_default_na=False, encoding="utf-8",
    )

### `collapse_duplicate_rows`

Duplicates are not allowed in the final cleaned dataset. Each dataset's raw export can contain multiple rows that agree on every field except its numeric measure(s) (e.g. the same `Date`/`CompanyCode`/`EventType`/... combination reported twice with different `UniqueUsers` counts) — this groups by every `ls_cols` field *except* `sum_cols`, sums `sum_cols` within each group, and re-sorts by `sort_cols` afterward (grouping does not guarantee the output is already in `SORT_COLS_*` order). Called once per section, right after `clean_*()` and before the cleaned CSV/report are written, so both reflect the deduplicated data.

In [ ]:
def collapse_duplicate_rows(
    df: pd.DataFrame, ls_cols: list[str], sum_cols: list[str], sort_cols: list[str]
) -> pd.DataFrame:
    group_cols = [c for c in ls_cols if c not in sum_cols]
    collapsed = df.groupby(group_cols, as_index=False, sort=False)[sum_cols].sum()[ls_cols]
    return collapsed.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)

## 1. MercuryDailyEvents

Cleans a raw `MercuryDailyEvents_yyyy-mm-dd.txt` export.

### Schema constants

In [6]:
LS_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "UniqueUsers", "DateRange",
]
LS_STRING_COLS_DE = [
    "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "DateRange",
]
LS_INT_COLS_DE = ["UniqueUsers"]

RENAME_MAP_DE = {
    "ContentTitleEN": "ContentTitle",
    "downloadLanguage": "DownloadLanguage",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType",
    "DeviceCategory", "SearchTerm", "ContentType", "ContentTitle",
]

PREFIX_DE = "MercuryDailyEvents"
FILENAME_RE_DE = re.compile(r"^MercuryDailyEvents_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `ContentTitleEN`→`ContentTitle`, `downloadLanguage`→`DownloadLanguage`, `uniqueUsers`→`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_DE` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DE`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DE` fields.
7. Cast `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_DE`.

In [7]:
def clean_de(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP_DE)

    # Same reasoning as the other notebooks: judge "completely blank" against the
    # LS_COLS fields present in the raw data, since raw pipeline-metadata columns
    # (FileName, PipelineRunID, ImportDate, CreatedBy, DataSource, ...) are dropped
    # later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS_DE if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_DE]

    for col in LS_STRING_COLS_DE:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DE:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DE, ascending=True).reset_index(drop=True)

    return df

### Configure the input file

Leave `INPUT_FILE_DE` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [8]:
INPUT_FILE_DE = None  # e.g. "input/MercuryDailyEvents_2026-08-02.txt"

input_path_de = Path(INPUT_FILE_DE).resolve() if INPUT_FILE_DE else find_default_input(INPUT_DIR, PREFIX_DE, FILENAME_RE_DE)
month_tag_de = month_tag_from_filename(input_path_de, PREFIX_DE, FILENAME_RE_DE)
input_path_de, month_tag_de

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryDailyEvents_2026-08-02.txt'),
 '202607')

### Read the raw text file

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the `\"..\"` escapes in `SearchTerm`/`ContentTitle`, protected against corrupting a literal backslash elsewhere in the row).

In [9]:
df_raw_de = read_semicolon_csv_protecting_backslashes(input_path_de)
df_raw_de.shape

(1587, 21)

### Apply the cleaning steps

In [10]:
df_cleaned_de = clean_de(df_raw_de)
df_cleaned_de.head()

,Date,CompanyCode,CompanyName,Country,Operation,EventType,DeviceCategory,SearchTerm,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,download,Desktop,-,Infosheet,How to Prioritize Self-care,en,,Search,1,2026-07-01 - 2026-07-31
1,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,download,Desktop,-,Infosheet,How to Prioritize When Everything Feels Urgent,en,,Search,1,2026-07-01 - 2026-07-31
2,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,search,Desktop,n/a,-,-,-,,,1,2026-07-01 - 2026-07-31
3,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,view,Desktop,-,Infosheet,How to Prioritize When Everything Feels Urgent,-,"Prioritization,Change",Search,1,2026-07-01 - 2026-07-31
4,2026-07-01 00:00:00.000,BDMY,Becton Dickinson,Malta,Lyra Health Malaysia Sdn Bhd,search,Desktop,n/a,-,-,-,,,1,2026-07-01 - 2026-07-31


### Collapse duplicate rows

Groups by every `LS_COLS_DE` field except `UniqueUsers` and sums `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_DE`, since grouping doesn't preserve the earlier sort order).

In [ ]:
rows_before_collapse_de = len(df_cleaned_de)
df_cleaned_de = collapse_duplicate_rows(df_cleaned_de, LS_COLS_DE, LS_INT_COLS_DE, SORT_COLS_DE)

print(f"Duplicate rows collapsed: {rows_before_collapse_de - len(df_cleaned_de)}")
df_cleaned_de.head()

### Save the cleaned dataset

In [11]:
output_path_de = OUTPUT_DIR / f"{PREFIX_DE}_{month_tag_de}_cleaned.csv"
df_cleaned_de.to_csv(output_path_de, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_de)} rows -> {output_path_de}")

Cleaned 1587 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryDailyEvents_202607_cleaned.csv


### Write summary report

In [ ]:
report_path_de, report_text_de = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DE,
    month_tag=month_tag_de,
    input_path=input_path_de,
    df_raw=df_raw_de,
    df_cleaned=df_cleaned_de,
    output_path=output_path_de,
    rows_before_collapse=rows_before_collapse_de,
)

print(report_text_de)
print(f"Report written -> {report_path_de}")

## 2. MercuryDailyUsers

Cleans a raw `MercuryDailyUsers_yyyy-mm-dd.txt` export.

### Schema constants

In [13]:
LS_COLS_DU = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "Sessions", "UniqueUsers", "DateRange",
]
LS_STRING_COLS_DU = [
    "CompanyCode", "CompanyName", "Country", "Operation", "DateRange",
]
LS_INT_COLS_DU = ["Sessions", "UniqueUsers"]

SORT_COLS_DU = ["Date", "CompanyCode", "CompanyName", "Country", "Operation"]

PREFIX_DU = "MercuryDailyUsers"
FILENAME_RE_DU = re.compile(r"^MercuryDailyUsers_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename every raw column so its first letter is capitalised (`sessions`→`Sessions`, `uniqueUsers`→`UniqueUsers`, ...) — via a lambda rather than an explicit rename map.
2. Drop rows that are blank across every `LS_COLS_DU` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DU` fields.
7. Cast `Sessions`, `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_DU`.

In [14]:
def clean_du(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=lambda c: c[:1].upper() + c[1:] if c else c)

    # Same reasoning as the other notebooks: judge "completely blank" against the
    # LS_COLS fields present in the raw data, since raw pipeline-metadata columns
    # (FileName, PipelineRunID, ImportDate, CreatedBy, DataSource, newUniqueUsers, ...)
    # are dropped later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS_DU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_DU]

    for col in LS_STRING_COLS_DU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DU:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DU, ascending=True).reset_index(drop=True)

    return df

### Configure the input file

Leave `INPUT_FILE_DU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [15]:
INPUT_FILE_DU = None  # e.g. "input/MercuryDailyUsers_2026-08-02.txt"

input_path_du = Path(INPUT_FILE_DU).resolve() if INPUT_FILE_DU else find_default_input(INPUT_DIR, PREFIX_DU, FILENAME_RE_DU)
month_tag_du = month_tag_from_filename(input_path_du, PREFIX_DU, FILENAME_RE_DU)
input_path_du, month_tag_du

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryDailyUsers_2026-08-02.txt'),
 '202607')

### Read the raw text file

Plain read, no `engine`/`escapechar` — this file has no free-text content needing escaped quotes, and those options would only risk corrupting a literal backslash in `CompanyCode`/`CompanyName`.

In [16]:
df_raw_du = pd.read_csv(input_path_du, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_du.shape

(236, 14)

### Apply the cleaning steps

In [17]:
df_cleaned_du = clean_du(df_raw_du)
df_cleaned_du.head()

,Date,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,1,1,2026-07-01 - 2026-07-31
1,2026-07-01 00:00:00.000,BDMY,Becton Dickinson,Malta,Lyra Health Malaysia Sdn Bhd,1,1,2026-07-01 - 2026-07-31
2,2026-07-01 00:00:00.000,BEIERSDORF,BEIERSDORF,United Kingdom,Lyra Health International Ltd,1,1,2026-07-01 - 2026-07-31
3,2026-07-01 00:00:00.000,DABWATERHUBTEST,DAB Water (Test Preview),Italy,Lyra Health International Ltd,1,1,2026-07-01 - 2026-07-31
4,2026-07-01 00:00:00.000,DATALOGIC,Datalogic,Malta,Lyra Health International Ltd,1,1,2026-07-01 - 2026-07-31


### Collapse duplicate rows

Groups by every `LS_COLS_DU` field except `Sessions`/`UniqueUsers` and sums `Sessions`, `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_DU`, since grouping doesn't preserve the earlier sort order).

In [ ]:
rows_before_collapse_du = len(df_cleaned_du)
df_cleaned_du = collapse_duplicate_rows(df_cleaned_du, LS_COLS_DU, LS_INT_COLS_DU, SORT_COLS_DU)

print(f"Duplicate rows collapsed: {rows_before_collapse_du - len(df_cleaned_du)}")
df_cleaned_du.head()

### Save the cleaned dataset

In [18]:
output_path_du = OUTPUT_DIR / f"{PREFIX_DU}_{month_tag_du}_cleaned.csv"
df_cleaned_du.to_csv(output_path_du, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_du)} rows -> {output_path_du}")

Cleaned 236 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryDailyUsers_202607_cleaned.csv


### Write summary report

In [ ]:
report_path_du, report_text_du = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DU,
    month_tag=month_tag_du,
    input_path=input_path_du,
    df_raw=df_raw_du,
    df_cleaned=df_cleaned_du,
    output_path=output_path_du,
    rows_before_collapse=rows_before_collapse_du,
)

print(report_text_du)
print(f"Report written -> {report_path_du}")

## 3. MercuryMonthlyEvents

Cleans a raw `MercuryMonthlyEvents_yyyy-mm-dd.txt` export.

### Schema constants

Note the column order here (`EventType, SearchTerm, DeviceCategory`) intentionally differs from DailyEvents' order (`EventType, DeviceCategory, SearchTerm`) — faithful to the original notebooks, not a typo.

In [20]:
LS_COLS_ME = [
    "MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "SearchTerm",
    "DeviceCategory", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "UniqueUsers", "DateRange",
]
LS_STRING_COLS_ME = [
    "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "SearchTerm",
    "DeviceCategory", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "DateRange",
]
LS_INT_COLS_ME = ["UniqueUsers"]

RENAME_MAP_ME = {
    "yearMonth": "MonthDate",
    "ContentTitleEN": "ContentTitle",
    "downloadLanguage": "DownloadLanguage",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_ME = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "EventType"]

PREFIX_ME = "MercuryMonthlyEvents"
FILENAME_RE_ME = re.compile(r"^MercuryMonthlyEvents_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `yearMonth`→`MonthDate`, `ContentTitleEN`→`ContentTitle`, `downloadLanguage`→`DownloadLanguage`, `uniqueUsers`→`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_ME` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` (raw `"yyyy-mm"`, no day) to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds), defaulting to the 1st of the month.
5. Reorder/drop columns to match `LS_COLS_ME`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_ME` fields.
7. Cast `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_ME` (note: only 6 columns, unlike DailyEvents' 10 — preserved faithfully, not "fixed" to match).

In [21]:
def clean_me(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP_ME)

    # Same reasoning as the other notebooks: judge "completely blank" against the
    # LS_COLS fields present in the raw data, since raw pipeline-metadata columns
    # (FileName, PipelineRunID, ImportDate, CreatedBy, DataSource, UserType, ...)
    # are dropped later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS_ME if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["MonthDate"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # Raw MonthDate values are "yyyy-mm" (no day); %m-only parsing defaults the day to 1.
    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_ME]

    for col in LS_STRING_COLS_ME:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_ME:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_ME, ascending=True).reset_index(drop=True)

    return df

### Configure the input file

Leave `INPUT_FILE_ME` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [22]:
INPUT_FILE_ME = None  # e.g. "input/MercuryMonthlyEvents_2026-08-02.txt"

input_path_me = Path(INPUT_FILE_ME).resolve() if INPUT_FILE_ME else find_default_input(INPUT_DIR, PREFIX_ME, FILENAME_RE_ME)
month_tag_me = month_tag_from_filename(input_path_me, PREFIX_ME, FILENAME_RE_ME)
input_path_me, month_tag_me

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryMonthlyEvents_2026-08-02.txt'),
 '202607')

### Read the raw text file

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the `\"..\"` escapes in `SearchTerm`/`ContentTitle`, protected against corrupting a literal backslash elsewhere in the row).

In [23]:
df_raw_me = read_semicolon_csv_protecting_backslashes(input_path_me)
df_raw_me.shape

(1404, 21)

### Apply the cleaning steps

In [24]:
df_cleaned_me = clean_me(df_raw_me)
df_cleaned_me.head()

,MonthDate,CompanyCode,CompanyName,Country,Operation,EventType,SearchTerm,DeviceCategory,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-07-01 00:00:00.000,ABPIE,ABP - Ireland,Ireland,Lyra UK & Ireland Ltd,search,n/a,Desktop,-,-,-,,,1,2026-07-01 - 2026-07-31
1,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,download,-,Desktop,Infosheet,How to Prioritize When Everything Feels Urgent,en,,Search,1,2026-07-01 - 2026-07-31
2,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,download,-,Desktop,Infosheet,How to Prioritize Self-care,en,,Search,1,2026-07-01 - 2026-07-31
3,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,search,n/a,Desktop,-,-,-,,,1,2026-07-01 - 2026-07-31
4,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,view,-,Desktop,Infosheet,How to Prioritize When Everything Feels Urgent,-,"Prioritization,Change",Search,1,2026-07-01 - 2026-07-31


### Collapse duplicate rows

Groups by every `LS_COLS_ME` field except `UniqueUsers` and sums `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_ME`, since grouping doesn't preserve the earlier sort order).

In [ ]:
rows_before_collapse_me = len(df_cleaned_me)
df_cleaned_me = collapse_duplicate_rows(df_cleaned_me, LS_COLS_ME, LS_INT_COLS_ME, SORT_COLS_ME)

print(f"Duplicate rows collapsed: {rows_before_collapse_me - len(df_cleaned_me)}")
df_cleaned_me.head()

### Save the cleaned dataset

In [25]:
output_path_me = OUTPUT_DIR / f"{PREFIX_ME}_{month_tag_me}_cleaned.csv"
df_cleaned_me.to_csv(output_path_me, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_me)} rows -> {output_path_me}")

Cleaned 1404 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryMonthlyEvents_202607_cleaned.csv


### Write summary report

In [ ]:
report_path_me, report_text_me = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_ME,
    month_tag=month_tag_me,
    input_path=input_path_me,
    df_raw=df_raw_me,
    df_cleaned=df_cleaned_me,
    output_path=output_path_me,
    rows_before_collapse=rows_before_collapse_me,
)

print(report_text_me)
print(f"Report written -> {report_path_me}")

## 4. MercuryMonthlyUsers

Cleans a raw `MercuryMonthlyUsers_yyyy-mm-dd.txt` export.

### Schema constants

In [27]:
LS_COLS_MU = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "Sessions", "UniqueUsers", "DateRange"]
LS_STRING_COLS_MU = ["CompanyCode", "CompanyName", "Country", "Operation", "DateRange"]
LS_INT_COLS_MU = ["Sessions", "UniqueUsers"]

RENAME_MAP_MU = {
    "yearMonth": "MonthDate",
    "sessions": "Sessions",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_MU = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation"]

PREFIX_MU = "MercuryMonthlyUsers"
FILENAME_RE_MU = re.compile(r"^MercuryMonthlyUsers_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `yearMonth`→`MonthDate`, `sessions`→`Sessions`, `uniqueUsers`→`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_MU` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` (raw `"yyyy-mm"`, no day) to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds), defaulting to the 1st of the month.
5. Reorder/drop columns to match `LS_COLS_MU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_MU` fields.
7. Cast `Sessions`, `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_MU`.

In [28]:
def clean_mu(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP_MU)

    present_ls_cols = [c for c in LS_COLS_MU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["MonthDate"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # Raw MonthDate values are "yyyy-mm" (no day); %m-only parsing defaults the day to 1.
    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_MU]

    for col in LS_STRING_COLS_MU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_MU:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_MU, ascending=True).reset_index(drop=True)

    return df

### Configure the input file

Leave `INPUT_FILE_MU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [29]:
INPUT_FILE_MU = None  # e.g. "input/MercuryMonthlyUsers_2026-08-02.txt"

input_path_mu = Path(INPUT_FILE_MU).resolve() if INPUT_FILE_MU else find_default_input(INPUT_DIR, PREFIX_MU, FILENAME_RE_MU)
month_tag_mu = month_tag_from_filename(input_path_mu, PREFIX_MU, FILENAME_RE_MU)
input_path_mu, month_tag_mu

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryMonthlyUsers_2026-08-02.txt'),
 '202607')

### Read the raw text file

Plain read, no `engine`/`escapechar` — this file has no free-text content needing escaped quotes, and those options would only risk corrupting a literal backslash in `CompanyCode`/`CompanyName`.

In [30]:
df_raw_mu = pd.read_csv(input_path_mu, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_mu.shape

(108, 14)

### Apply the cleaning steps

In [31]:
df_cleaned_mu = clean_mu(df_raw_mu)
df_cleaned_mu.head()

,MonthDate,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-07-01 00:00:00.000,ABPIE,ABP - Ireland,Ireland,Lyra UK & Ireland Ltd,3,1,2026-07-01 - 2026-07-31
1,2026-07-01 00:00:00.000,ABPUK,ABP - UK,United Kingdom,Lyra UK & Ireland Ltd,1,1,2026-07-01 - 2026-07-31
2,2026-07-01 00:00:00.000,ADV,ABP - Advanced Proteins,United Kingdom,Lyra UK & Ireland Ltd,2,1,2026-07-01 - 2026-07-31
3,2026-07-01 00:00:00.000,ASAHI,ASAHI,Italy,Lyra Health International Ltd,1,1,2026-07-01 - 2026-07-31
4,2026-07-01 00:00:00.000,AVANTOR,Avantor,Belgium,Lyra Health International Ltd,1,1,2026-07-01 - 2026-07-31


### Collapse duplicate rows

Groups by every `LS_COLS_MU` field except `Sessions`/`UniqueUsers` and sums `Sessions`, `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_MU`, since grouping doesn't preserve the earlier sort order).

In [ ]:
rows_before_collapse_mu = len(df_cleaned_mu)
df_cleaned_mu = collapse_duplicate_rows(df_cleaned_mu, LS_COLS_MU, LS_INT_COLS_MU, SORT_COLS_MU)

print(f"Duplicate rows collapsed: {rows_before_collapse_mu - len(df_cleaned_mu)}")
df_cleaned_mu.head()

### Save the cleaned dataset

In [32]:
output_path_mu = OUTPUT_DIR / f"{PREFIX_MU}_{month_tag_mu}_cleaned.csv"
df_cleaned_mu.to_csv(output_path_mu, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_mu)} rows -> {output_path_mu}")

Cleaned 108 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryMonthlyUsers_202607_cleaned.csv


### Write summary report

In [ ]:
report_path_mu, report_text_mu = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_MU,
    month_tag=month_tag_mu,
    input_path=input_path_mu,
    df_raw=df_raw_mu,
    df_cleaned=df_cleaned_mu,
    output_path=output_path_mu,
    rows_before_collapse=rows_before_collapse_mu,
)

print(report_text_mu)
print(f"Report written -> {report_path_mu}")

## Summary

Convenience recap of everything produced by this run.

In [34]:
print("Cleaned outputs:")
for label, out_path, rpt_path in [
    ("DailyEvents",   output_path_de, report_path_de),
    ("DailyUsers",    output_path_du, report_path_du),
    ("MonthlyEvents", output_path_me, report_path_me),
    ("MonthlyUsers",  output_path_mu, report_path_mu),
]:
    print(f"  {label:14s} -> {out_path.name}  (report: {rpt_path.name})")

Cleaned outputs:
  DailyEvents    -> MercuryDailyEvents_202607_cleaned.csv  (report: MercuryDailyEvents_202607_report.txt)
  DailyUsers     -> MercuryDailyUsers_202607_cleaned.csv  (report: MercuryDailyUsers_202607_report.txt)
  MonthlyEvents  -> MercuryMonthlyEvents_202607_cleaned.csv  (report: MercuryMonthlyEvents_202607_report.txt)
  MonthlyUsers   -> MercuryMonthlyUsers_202607_cleaned.csv  (report: MercuryMonthlyUsers_202607_report.txt)
